# QAI Validation Evaluation

This notebook reads a QAI training run from `saves/run_artifacts/train_qai`, supports both flat and nested artifact layouts, and reruns the checkpoint on the validation split to regenerate prediction outputs.

Generated validation artifacts are written back into the run folder:
- `validation_predictions.csv`
- `validation_prediction_metrics.json`
- `validation_price_prediction.png`


In [ ]:
from pathlib import Path

RUN_DIR = Path("saves/run_artifacts/train_qai")
EVAL_SPLIT = "val"
CHECKPOINT_FILENAME = "best_checkpoint.pt"
EPOCH_METRICS_FILENAME = "epoch_metrics.json"
PREDICTIONS_FILENAME = "validation_predictions.csv"
PREDICTION_METRICS_FILENAME = "validation_prediction_metrics.json"
PRICE_PLOT_FILENAME = "validation_price_prediction.png"
PLOT_HORIZON = 1
SAMPLE_PRICE_PREDICTION = 6
RANDOM_SAMPLE_SEED = 123
WRITE_OUTPUTS = True


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists():
    for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
        if (candidate / "src").exists() and (candidate / "pyproject.toml").exists():
            REPO_ROOT = candidate
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from src.datamodules.qai_datamodule import QaiDataModule
from src.models.qai_attention import QaiAttentionModel

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError(
        "matplotlib is required to run this notebook. Install it in the environment first."
    ) from exc

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the current working directory.")


def resolve_run_dir(run_dir: str | Path) -> Path:
    repo_root = find_repo_root()
    run_path = Path(run_dir)
    return run_path if run_path.is_absolute() else (repo_root / run_path).resolve()


def load_json(path: Path):
    if not path.exists():
        return None
    return json.loads(path.read_text())


def first_existing_path(run_dir: Path, *relative_paths: str) -> Path | None:
    for relative_path in relative_paths:
        candidate = run_dir / relative_path
        if candidate.exists():
            return candidate
    return None


def require_existing_path(run_dir: Path, *relative_paths: str) -> Path:
    candidate = first_existing_path(run_dir, *relative_paths)
    if candidate is None:
        raise FileNotFoundError(
            f"Could not find any of the expected files under {run_dir}: {relative_paths}"
        )
    return candidate


def load_named_json(run_dir: Path, filename: str):
    path = first_existing_path(run_dir, filename, f"artifacts/{filename}")
    return load_json(path) if path is not None else None


def load_epoch_frame(run_dir: Path, epoch_metrics_filename: str) -> pd.DataFrame:
    epoch_metrics_path = run_dir / epoch_metrics_filename
    if epoch_metrics_path.exists():
        frame = pd.DataFrame(load_json(epoch_metrics_path))
    else:
        raise FileNotFoundError(
            f"Missing saved epoch history. Expected {epoch_metrics_path}"
        )
    if "epoch" in frame.columns:
        frame = frame.sort_values("epoch").reset_index(drop=True)
    return frame


def build_snapshot_epoch_frame(*records) -> pd.DataFrame:
    usable_records = [record for record in records if isinstance(record, dict) and record]
    if not usable_records:
        return pd.DataFrame()
    frame = pd.DataFrame(usable_records)
    if "epoch" in frame.columns:
        frame = frame.sort_values("epoch").reset_index(drop=True)
    return frame


def absolutize_path(path_value: str | Path, repo_root: Path) -> str:
    path_obj = Path(path_value)
    return str(path_obj if path_obj.is_absolute() else (repo_root / path_obj).resolve())


def build_datamodule(training_setup: dict) -> QaiDataModule:
    repo_root = find_repo_root()
    data_cfg = dict(training_setup.get("data", {}))
    data_cfg.pop("_target_", None)
    for key in ["train_path", "val_path", "test_path", "price_history_path"]:
        if key in data_cfg and data_cfg[key] is not None:
            data_cfg[key] = absolutize_path(data_cfg[key], repo_root)
    return QaiDataModule(**data_cfg)


def build_model(training_setup: dict, datamodule: QaiDataModule) -> QaiAttentionModel:
    model_cfg = dict(training_setup.get("model", {}))
    model_cfg.pop("_target_", None)
    model_cfg["input_dim"] = datamodule.feature_dim
    model_cfg["sequence_length"] = max(
        int(model_cfg.get("sequence_length", datamodule.max_sequence_length)),
        int(datamodule.max_sequence_length),
    )
    return QaiAttentionModel(**model_cfg)


def checkpoint_model_state(checkpoint: dict) -> dict:
    if "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    return checkpoint


def build_validation_predictions(
    run_dir: Path,
    training_setup: dict,
    checkpoint_filename: str,
    eval_split: str,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    if eval_split != "val":
        raise ValueError(f"This notebook is configured for validation evaluation. Received split={eval_split!r}.")

    checkpoint_path = require_existing_path(run_dir, checkpoint_filename, f"artifacts/{checkpoint_filename}")
    datamodule = build_datamodule(training_setup)
    datamodule.setup()

    model = build_model(training_setup, datamodule)
    checkpoint = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(checkpoint_model_state(checkpoint))
    model.eval()

    rows: list[dict[str, object]] = []
    with torch.no_grad():
        for batch in datamodule.val_dataloader():
            outputs = model(batch["features"], attention_mask=batch["attention_mask"])
            price_preds = outputs["price_preds"].detach().cpu().numpy()
            class_logits = outputs["class_logits"].detach().cpu().numpy()
            class_preds = class_logits.argmax(axis=-1)
            price_targets = batch["price_targets"].detach().cpu().numpy()
            class_targets = batch["class_targets"].detach().cpu().numpy()
            sequence_lengths = batch["sequence_lengths"].detach().cpu().numpy()

            for sample_idx in range(price_preds.shape[0]):
                for horizon_idx in range(price_preds.shape[1]):
                    rows.append(
                        {
                            "split": eval_split,
                            "ticker": batch["ticker"][sample_idx],
                            "time_range": batch["time_range"][sample_idx],
                            "quarter_end_date": pd.Timestamp(batch["quarter_end_date"][sample_idx]),
                            "sequence_length": int(sequence_lengths[sample_idx]),
                            "horizon": horizon_idx + 1,
                            "predicted_price": float(price_preds[sample_idx, horizon_idx]),
                            "target_price": float(price_targets[sample_idx, horizon_idx]),
                            "price_error": float(price_preds[sample_idx, horizon_idx] - price_targets[sample_idx, horizon_idx]),
                            "predicted_class": int(class_preds[sample_idx, horizon_idx]),
                            "target_class": int(class_targets[sample_idx, horizon_idx]),
                        }
                    )

    predictions = pd.DataFrame(rows)
    predictions = predictions.sort_values(["quarter_end_date", "ticker", "horizon"]).reset_index(drop=True)
    predictions["absolute_error"] = predictions["price_error"].abs()
    predictions["correct_class"] = predictions["predicted_class"] == predictions["target_class"]

    summary_rows = []
    for horizon, horizon_df in predictions.groupby("horizon", sort=True):
        summary_rows.append(
            {
                "split": eval_split,
                "horizon": int(horizon),
                "rows": int(len(horizon_df)),
                "price_mae": float(mean_absolute_error(horizon_df["target_price"], horizon_df["predicted_price"])),
                "price_rmse": float(np.sqrt(mean_squared_error(horizon_df["target_price"], horizon_df["predicted_price"]))),
                "price_r2": float(r2_score(horizon_df["target_price"], horizon_df["predicted_price"])),
                "class_accuracy": float(horizon_df["correct_class"].mean()),
                "mean_target_price": float(horizon_df["target_price"].mean()),
                "mean_predicted_price": float(horizon_df["predicted_price"].mean()),
            }
        )

    summary = pd.DataFrame(summary_rows)
    overall_metrics = {
        "split": eval_split,
        "rows": int(len(predictions)),
        "sample_count": int(predictions[["ticker", "quarter_end_date"]].drop_duplicates().shape[0]),
        "price_mae": float(mean_absolute_error(predictions["target_price"], predictions["predicted_price"])),
        "price_rmse": float(np.sqrt(mean_squared_error(predictions["target_price"], predictions["predicted_price"]))),
        "price_r2": float(r2_score(predictions["target_price"], predictions["predicted_price"])),
        "class_accuracy": float(predictions["correct_class"].mean()),
    }
    return predictions, summary, overall_metrics


RUN_DIR = resolve_run_dir(RUN_DIR)
RUN_DIR


In [ ]:
training_setup = load_named_json(RUN_DIR, "training_setup.json")
final_metrics = load_named_json(RUN_DIR, "metrics.json")
best_metrics = load_named_json(RUN_DIR, "best_metrics.json")
best_checkpoint_summary = load_named_json(RUN_DIR, "best_checkpoint_summary.json")
hyperparams = load_named_json(RUN_DIR, "training_hyperparameters.json")
data_summary = pd.DataFrame(load_named_json(RUN_DIR, "data_summary.json") or [])
model_structure = load_named_json(RUN_DIR, "model_structure.json")

try:
    epoch_df = load_epoch_frame(RUN_DIR, EPOCH_METRICS_FILENAME)
    epoch_history_source = EPOCH_METRICS_FILENAME
except FileNotFoundError:
    epoch_df = build_snapshot_epoch_frame(best_metrics, final_metrics, best_checkpoint_summary)
    epoch_history_source = "best/final snapshot artifacts"

if training_setup is None:
    raise FileNotFoundError("training_setup.json is required to rebuild validation predictions.")
if epoch_df.empty:
    raise FileNotFoundError("No epoch history or snapshot metrics were found for this run.")

display(Markdown(f"## Loaded Run: `{RUN_DIR}`"))
display(Markdown(f"Epoch rows loaded: **{len(epoch_df)}** from **{epoch_history_source}**"))
epoch_df.head()


In [ ]:
summary_rows = []
if hyperparams:
    summary_rows.append({"section": "training_hyperparameters", **hyperparams})
if final_metrics:
    summary_rows.append({"section": "final_metrics", **final_metrics})
if best_metrics:
    summary_rows.append({"section": "best_metrics", **best_metrics})

config_rows = [
    {
        "run_dir": str(RUN_DIR),
        "eval_split": EVAL_SPLIT,
        "checkpoint_filename": CHECKPOINT_FILENAME,
        "epoch_metrics_filename": EPOCH_METRICS_FILENAME,
        "predictions_filename": PREDICTIONS_FILENAME,
        "prediction_metrics_filename": PREDICTION_METRICS_FILENAME,
        "price_plot_filename": PRICE_PLOT_FILENAME,
        "plot_horizon": PLOT_HORIZON,
        "sample_price_prediction": SAMPLE_PRICE_PREDICTION,
        "random_sample_seed": RANDOM_SAMPLE_SEED,
        "write_outputs": WRITE_OUTPUTS,
    }
]

display(Markdown("## Notebook Configuration"))
display(pd.DataFrame(config_rows))

display(Markdown("## Run Tables"))
if not data_summary.empty:
    display(Markdown("### Data Summary"))
    display(data_summary)

if summary_rows:
    display(Markdown("### Key Run Metadata"))
    display(pd.DataFrame(summary_rows))

if model_structure:
    display(Markdown("### Model Structure"))
    display(pd.json_normalize(model_structure, sep="."))


In [ ]:
best_epoch = None
if "val_loss" in epoch_df.columns and not epoch_df["val_loss"].isna().all():
    best_epoch = epoch_df.loc[epoch_df["val_loss"].idxmin(), "epoch"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epoch_df["epoch"], epoch_df["train_loss"], marker="o", label="train")
axes[0].plot(epoch_df["epoch"], epoch_df["val_loss"], marker="o", label="val")
axes[0].set_title("Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(epoch_df["epoch"], epoch_df["train_price_mae"], marker="o", label="train")
axes[1].plot(epoch_df["epoch"], epoch_df["val_price_mae"], marker="o", label="val")
axes[1].set_title("Price MAE by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("MAE")
axes[1].legend()

axes[2].plot(epoch_df["epoch"], epoch_df["train_class_accuracy"], marker="o", label="train")
axes[2].plot(epoch_df["epoch"], epoch_df["val_class_accuracy"], marker="o", label="val")
axes[2].set_title("Classification Accuracy by Epoch")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Accuracy")
axes[2].legend()

if best_epoch is not None:
    for ax in axes:
        ax.axvline(best_epoch, color="tab:red", linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(epoch_df["epoch"], epoch_df["train_regression_loss"], marker="o", label="train regression")
axes[0].plot(epoch_df["epoch"], epoch_df["val_regression_loss"], marker="o", label="val regression")
axes[0].plot(epoch_df["epoch"], epoch_df["train_classification_loss"], marker="o", label="train classification")
axes[0].plot(epoch_df["epoch"], epoch_df["val_classification_loss"], marker="o", label="val classification")
axes[0].set_title("Loss Components")
axes[0].set_xlabel("Epoch")
axes[0].legend()

epoch_df["loss_gap"] = epoch_df["val_loss"] - epoch_df["train_loss"]
epoch_df["mae_gap"] = epoch_df["val_price_mae"] - epoch_df["train_price_mae"]
epoch_df["accuracy_gap"] = epoch_df["train_class_accuracy"] - epoch_df["val_class_accuracy"]

axes[1].plot(epoch_df["epoch"], epoch_df["loss_gap"], marker="o", label="val_loss - train_loss")
axes[1].plot(epoch_df["epoch"], epoch_df["mae_gap"], marker="o", label="val_mae - train_mae")
axes[1].plot(epoch_df["epoch"], epoch_df["accuracy_gap"], marker="o", label="train_acc - val_acc")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set_title("Generalization Gaps")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

train_horizon_cols = [column for column in epoch_df.columns if column.startswith("train_class_accuracy_h")]
val_horizon_cols = [column for column in epoch_df.columns if column.startswith("val_class_accuracy_h")]

fig, axes = plt.subplots(1, 2, figsize=(18, 5), sharey=True)
for column in train_horizon_cols:
    axes[0].plot(epoch_df["epoch"], epoch_df[column], marker="o", label=column.replace("train_", ""))
axes[0].set_title("Train Accuracy by Horizon")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

for column in val_horizon_cols:
    axes[1].plot(epoch_df["epoch"], epoch_df[column], marker="o", label=column.replace("val_", ""))
axes[1].set_title("Validation Accuracy by Horizon")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
predictions_df, horizon_summary_df, validation_metrics = build_validation_predictions(
    run_dir=RUN_DIR,
    training_setup=training_setup,
    checkpoint_filename=CHECKPOINT_FILENAME,
    eval_split=EVAL_SPLIT,
)

if WRITE_OUTPUTS:
    predictions_output_path = RUN_DIR / PREDICTIONS_FILENAME
    metrics_output_path = RUN_DIR / PREDICTION_METRICS_FILENAME
    predictions_df.to_csv(predictions_output_path, index=False)
    metrics_output_path.write_text(json.dumps({
        "overall": validation_metrics,
        "by_horizon": horizon_summary_df.to_dict(orient="records"),
    }, indent=2))
else:
    predictions_output_path = None
    metrics_output_path = None

display(Markdown("## Validation Prediction Outputs"))
display(pd.DataFrame([validation_metrics]))
display(horizon_summary_df)
display(predictions_df.head())


In [ ]:
overview_df = predictions_df.loc[predictions_df["horizon"] == PLOT_HORIZON].copy()
overview_df = overview_df.sort_values(["quarter_end_date", "ticker"]).reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(predictions_df["target_price"], predictions_df["predicted_price"], alpha=0.6)
diagonal_min = float(min(predictions_df["target_price"].min(), predictions_df["predicted_price"].min()))
diagonal_max = float(max(predictions_df["target_price"].max(), predictions_df["predicted_price"].max()))
axes[0].plot([diagonal_min, diagonal_max], [diagonal_min, diagonal_max], color="tab:red", linestyle="--")
axes[0].set_title("Validation Predicted vs Actual")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")

axes[1].hist(predictions_df["price_error"], bins=30, color="tab:blue", alpha=0.8)
axes[1].set_title("Validation Price Error Distribution")
axes[1].set_xlabel("Predicted - Actual")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

sample_keys = (
    predictions_df[["ticker", "time_range", "quarter_end_date", "sequence_length"]]
    .drop_duplicates()
    .sort_values(["quarter_end_date", "ticker"])
    .reset_index(drop=True)
)
sample_count = min(SAMPLE_PRICE_PREDICTION, len(sample_keys))
sampled_keys = sample_keys.sample(n=sample_count, random_state=RANDOM_SAMPLE_SEED).reset_index(drop=True)
sampled_predictions = sampled_keys.merge(
    predictions_df,
    on=["ticker", "time_range", "quarter_end_date", "sequence_length"],
    how="left",
).sort_values(["quarter_end_date", "ticker", "horizon"]).reset_index(drop=True)

sample_stats_rows = []
for _, sample_key in sampled_keys.iterrows():
    sample_frame = sampled_predictions.loc[
        (sampled_predictions["ticker"] == sample_key["ticker"])
        & (sampled_predictions["time_range"] == sample_key["time_range"])
        & (sampled_predictions["quarter_end_date"] == sample_key["quarter_end_date"])
        & (sampled_predictions["sequence_length"] == sample_key["sequence_length"])
    ].sort_values("horizon")
    sample_stats_rows.append(
        {
            "ticker": sample_key["ticker"],
            "time_range": sample_key["time_range"],
            "quarter_end_date": sample_key["quarter_end_date"],
            "sequence_length": sample_key["sequence_length"],
            "mae": float(mean_absolute_error(sample_frame["target_price"], sample_frame["predicted_price"])),
            "rmse": float(np.sqrt(mean_squared_error(sample_frame["target_price"], sample_frame["predicted_price"]))),
            "mean_actual_price": float(sample_frame["target_price"].mean()),
            "mean_predicted_price": float(sample_frame["predicted_price"].mean()),
        }
    )
sample_stats_df = pd.DataFrame(sample_stats_rows)
sample_stats_df.insert(0, "sample_id", np.arange(1, len(sample_stats_df) + 1))

display(Markdown("## Random Validation Sample Stats"))
display(sample_stats_df)

subplot_cols = 2 if sample_count > 1 else 1
subplot_rows = int(np.ceil(sample_count / subplot_cols))
fig, axes = plt.subplots(subplot_rows, subplot_cols, figsize=(7 * subplot_cols, 4.5 * subplot_rows), squeeze=False)
axes_flat = axes.flatten()

for axis_idx, (_, sample_key) in enumerate(sampled_keys.iterrows()):
    ax = axes_flat[axis_idx]
    sample_frame = sampled_predictions.loc[
        (sampled_predictions["ticker"] == sample_key["ticker"])
        & (sampled_predictions["time_range"] == sample_key["time_range"])
        & (sampled_predictions["quarter_end_date"] == sample_key["quarter_end_date"])
        & (sampled_predictions["sequence_length"] == sample_key["sequence_length"])
    ].sort_values("horizon")
    sample_stats = sample_stats_df.iloc[axis_idx]
    ax.plot(sample_frame["horizon"], sample_frame["target_price"], marker="o", linewidth=2, label="actual")
    ax.plot(sample_frame["horizon"], sample_frame["predicted_price"], marker="o", linewidth=2, label="predicted")
    ax.set_title(
        f"Sample {axis_idx + 1}: {sample_key['ticker']} | {sample_key['time_range']}\n"
        f"MAE={sample_stats['mae']:.2f}, RMSE={sample_stats['rmse']:.2f}"
    )
    ax.set_xlabel("Horizon")
    ax.set_ylabel("Price")
    ax.set_xticks(sample_frame["horizon"].tolist())
    ax.legend()

for axis_idx in range(sample_count, len(axes_flat)):
    axes_flat[axis_idx].axis("off")

plt.tight_layout()

price_plot_path = RUN_DIR / PRICE_PLOT_FILENAME
if WRITE_OUTPUTS:
    fig.savefig(price_plot_path, dpi=200, bbox_inches="tight")
plt.show()

display(Markdown("## Random Validation Sample Predictions"))
display(sampled_predictions[["ticker", "time_range", "quarter_end_date", "sequence_length", "horizon", "target_price", "predicted_price", "price_error", "absolute_error"]])


In [ ]:
display(Markdown("## Best Epoch Snapshot"))
if best_checkpoint_summary is not None:
    display(pd.DataFrame([best_checkpoint_summary]))
elif best_metrics is not None:
    display(pd.DataFrame([best_metrics]))
else:
    best_epoch_row = epoch_df.loc[[epoch_df["val_loss"].idxmin()]] if "val_loss" in epoch_df.columns else epoch_df.tail(1)
    display(best_epoch_row)

display(Markdown("## Last 10 Epochs"))
display(epoch_df.tail(10))

output_rows = [{
    "predictions_output": str(predictions_output_path) if predictions_output_path else None,
    "prediction_metrics_output": str(metrics_output_path) if metrics_output_path else None,
    "price_plot_output": str(RUN_DIR / PRICE_PLOT_FILENAME) if WRITE_OUTPUTS else None,
}]
display(Markdown("## Saved Output Paths"))
display(pd.DataFrame(output_rows))
